# M3_D1 — Operasi Dasar Pengolahan Citra

Modul 3 — Pengolahan Citra dan Visi Komputer. Praktikum D1 (Operasi Citra Sederhana) + Tugas Praktikum 1-11.

## D1. Operasi Citra Sederhana

### Langkah 1 — Setup, Import Library, dan Mount Google Drive

In [1]:
import os
import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    def cv2_imshow(img):
        img_rgb = cv.cvtColor(img, cv.COLOR_BGR2RGB) if img.ndim == 3 else img
        plt.imshow(img_rgb, cmap='gray' if img.ndim == 2 else None)
        plt.axis('off')
        plt.show()

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Bukan di Colab, skip mount drive (lagi dikerjakan lokal).')

Mounted at /content/drive


### Langkah 2 — Menyiapkan Citra


In [ ]:
os.makedirs('images', exist_ok=True)
os.makedirs('images/noises', exist_ok=True)

from skimage import data as skdata

def ensure(path, builder):
    if not os.path.exists(path):
        cv.imwrite(path, builder())
    return path


def make_houses():
    return cv.cvtColor(skdata.astronaut(), cv.COLOR_RGB2BGR)


def make_peppers():
    img = np.full((450, 650, 3), 255, np.uint8)
    cv.circle(img, (150, 260), 130, (40, 180, 40), -1)   # hijau
    cv.circle(img, (330, 280), 140, (30, 30, 210), -1)   # merah
    cv.circle(img, (500, 260), 130, (30, 200, 230), -1)  # kuning
    return img


def make_galaxy():
    h, w = 200, 200
    yy, xx = np.mgrid[0:h, 0:w]
    r = np.sqrt((yy - h / 2) ** 2 + (xx - w / 2) ** 2)
    glow = np.clip(255 - r * 1.6, 0, 255).astype(np.uint8)
    return cv.merge([(glow * 0.6).astype(np.uint8), (glow * 0.7).astype(np.uint8), glow])


def make_couple():
    astro = cv.cvtColor(skdata.astronaut(), cv.COLOR_RGB2BGR)
    face = astro[0:280, 100:380]
    return cv.hconcat([face, face])


def make_crayfish():
    rng = np.random.default_rng(1)
    base = rng.integers(20, 70, (300, 450, 3), dtype=np.uint8)
    base[:, :, 1] = np.clip(base[:, :, 1].astype(int) + 20, 0, 255).astype(np.uint8)
    img = cv.GaussianBlur(base, (15, 15), 0)
    cv.ellipse(img, (230, 170), (70, 30), 20, 0, 360, (30, 60, 150), -1)
    return img


def make_night():
    astro = cv.cvtColor(skdata.astronaut(), cv.COLOR_RGB2BGR)
    return np.clip(astro.astype(np.float32) * 0.12, 0, 255).astype(np.uint8)


ensure('images/houses.jpg', make_houses)
ensure('images/peppers.jpg', make_peppers)
ensure('images/galaxy.jpg', make_galaxy)
ensure('images/couple.jpg', make_couple)
ensure('images/crayfish.jpg', make_crayfish)
ensure('images/night.jpg', make_night)

if len(glob.glob('images/noises/*.jpg')) < 100:
    galaxy_base = cv.imread('images/galaxy.jpg')
    rng = np.random.default_rng(42)
    for i in range(100):
        noise = rng.normal(0, 25, galaxy_base.shape)
        noisy = np.clip(galaxy_base.astype(np.float32) + noise, 0, 255).astype(np.uint8)
        cv.imwrite(f'images/noises/noise_{i:03d}.jpg', noisy)

print('Semua citra sample siap di folder images/. Ganti path ke citra asli Anda kalau sudah ada.')

### Langkah 3 — Transformasi Linier Brightness

Formula: `g(x,y) = f(x,y) + b`, dengan `g(x,y)` nilai pixel setelah transformasi, `f(x,y)` nilai pixel asli, dan `b` nilai brightness.

In [ ]:
print(' Mengubah tingkat kecerahan citra ')
print('-----------------------------------')
try:
    brightness = int(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number')

# sesuaikan dengan file dan folder di drive Anda
original = cv.imread('images/houses.jpg')
brightness_image = np.zeros(original.shape, original.dtype)

# akses per piksel
# np.clip digunakan untuk melakukan truncate pixel (nilai dibatasi antara 0-255)
for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for c in range(original.shape[2]):
            brightness_image[y, x, c] = np.clip(original[y, x, c] + brightness, 0, 255)

# cara simple tanpa for loop
# brightness_image = cv.convertScaleAbs(original, beta=brightness)

final_frame = cv.hconcat((original, brightness_image))
cv2_imshow(final_frame)

## TUGAS PRAKTIKUM

### 1. Inverse Citra

Formula: `g(x) = 255 - f(x)`.

In [ ]:
img = cv.imread('images/peppers.jpg')  # ganti ke path peppers.jpg Anda kalau sudah ada
inverse_image = 255 - img

result = cv.hconcat([img, inverse_image])
cv2_imshow(result)

### 2. Transformasi Contrast

Formula: `g(x,y) = a * f(x,y) + b`, dengan `a` nilai contrast dan `b` nilai brightness.

In [ ]:
print(' Mengubah kontras dan tingkat kecerahan citra ')
print('-----------------------------------------------')
try:
    brightness = int(input('Masukkan tingkat kecerahan: '))
    contrast = float(input('Masukkan kontras: '))
except ValueError:
    print('Error, not a number')

original = cv.imread('images/houses.jpg')
contrast_image = np.zeros(original.shape, original.dtype)

for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for c in range(original.shape[2]):
            contrast_image[y, x, c] = np.clip(contrast * original[y, x, c] + brightness, 0, 255)

final_frame = cv.hconcat((original, contrast_image))
cv2_imshow(final_frame)

### 3. Transformasi Logarithmic Brightness

Formula: `s = c * log(1 + r)`. Konstanta `c` dinormalisasi otomatis (`255 / log(256)`) supaya output tetap di range 0-255, lalu diskalakan dengan nilai kecerahan yang diinput user (referensi 50, sesuai contoh pada modul).

In [ ]:
print(' Mengubah tingkat kecerahan citra dengan Transformasi Log ')
print('-------------------------------------------------------------')
try:
    kecerahan = int(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number')

original = cv.imread('images/houses.jpg')
log_image = np.zeros(original.shape, original.dtype)

c = 255 / np.log(1 + 255)
scale = kecerahan / 50  # 50 dipakai sebagai nilai referensi pada contoh modul

for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for ch in range(original.shape[2]):
            s = c * np.log(1 + original[y, x, ch]) * scale
            log_image[y, x, ch] = np.clip(s, 0, 255)

final_frame = cv.hconcat((original, log_image))
cv2_imshow(final_frame)

### 4. Grayscale — Averaging, Lightness, Luminance

In [ ]:
img = cv.imread('images/peppers.jpg')
b, g, r = cv.split(img.astype(np.float32))

avg_gray = ((r + g + b) / 3).astype(np.uint8)
lightness_gray = (
    (np.maximum(np.maximum(r, g), b) + np.minimum(np.minimum(r, g), b)) / 2
).astype(np.uint8)
luminance_gray = (0.21 * r + 0.72 * g + 0.07 * b).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Original', 'Averaging', 'Lightness', 'Luminance']
hasil = [cv.cvtColor(img, cv.COLOR_BGR2RGB), avg_gray, lightness_gray, luminance_gray]
for ax, title, im in zip(axes, titles, hasil):
    ax.imshow(im, cmap=None if title == 'Original' else 'gray')
    ax.set_title(title)
plt.tight_layout()
plt.show()

### 5. Highlight Warna Tertentu, Sisanya Grayscale

Warna merah dipertahankan (dideteksi lewat HSV hue merah), bagian lain diubah jadi grayscale.

In [ ]:
img = cv.imread('images/peppers.jpg')
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
gray_3ch = cv.cvtColor(gray, cv.COLOR_GRAY2BGR)

hsv = cv.cvtColor(img, cv.COLOR_BGR2HSV)
mask_merah_1 = cv.inRange(hsv, (0, 90, 60), (10, 255, 255))
mask_merah_2 = cv.inRange(hsv, (170, 90, 60), (180, 255, 255))
mask_merah = cv.bitwise_or(mask_merah_1, mask_merah_2)
mask_merah_3ch = cv.cvtColor(mask_merah, cv.COLOR_GRAY2BGR)

result = np.where(mask_merah_3ch == 255, img, gray_3ch)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB)); axes[0].set_title('Original')
axes[1].imshow(cv.cvtColor(result, cv.COLOR_BGR2RGB)); axes[1].set_title('Merah dipertahankan, lainnya grayscale')
plt.show()

### 6. Gamma Correction

Formula: `I' = 255 * (I/255)^gamma`.

In [ ]:
print(' Gamma Correction pada citra ')
print('--------------------------------')
try:
    gamma = float(input('Masukkan nilai Gamma: '))
except ValueError:
    print('Error, not a number')

original = cv.imread('images/houses.jpg')
normalized = original.astype(np.float32) / 255.0
gamma_image = np.clip(255 * (normalized ** gamma), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv.cvtColor(original, cv.COLOR_BGR2RGB)); axes[0].set_title('Citra Asli')
axes[1].imshow(cv.cvtColor(gamma_image, cv.COLOR_BGR2RGB)); axes[1].set_title(f'Gamma Correction (γ = {gamma})')
plt.show()

### 7. Simulasi Image Depth

Formula: `level = 255 / (2^bit_depth - 1)`, lalu `C' = round(C / level) * level`.

In [ ]:
print('Simulasi Image Depth')
print('------------------------')
try:
    bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
except ValueError:
    print('Error, not a number')

print('Bit depth awal  : 8 bit')
print('Bit depth tujuan:', bit_depth, 'bit')
print('Jumlah level    :', pow(2, bit_depth))

level = 255 / (pow(2, bit_depth) - 1)
original = cv.imread('images/peppers.jpg', cv.IMREAD_GRAYSCALE)
depth_image = np.zeros(original.shape, original.dtype)

for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        depth_image[y, x] = np.clip(round(original[y, x] / level) * level, 0, 255)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original, cmap='gray'); axes[0].set_title('Grayscale 8-bit')
axes[1].imshow(depth_image, cmap='gray'); axes[1].set_title(f'Grayscale {bit_depth}-bit')
plt.show()

### 8. Average Denoising + PSNR

Citra asli: `images/galaxy.jpg` (sample sintetis). 100 citra ber-Gaussian Noise: `images/noises/*.jpg` (dibuat otomatis di Langkah 2). Nanti kalau sudah ada citra asli dari modul, tinggal timpa kedua path itu.

In [ ]:
def PSNR(img1, img2):
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:  # MSE 0 maka tidak ada noise sama sekali, sehingga PSNR tidak memiliki arti
        return 100
    max_pixel = 255.0
    return 20 * np.log10(max_pixel / np.sqrt(mse))


original_galaxy = cv.imread('images/galaxy.jpg')

cv_img = []
for path in sorted(glob.glob('images/noises/*.jpg')):
    n = cv.imread(path)
    cv_img.append(n)

jumlah_list = [5, 10, 20, 40, 80, 100]
results = []
for jumlah in jumlah_list:
    stacked = np.stack(cv_img[:jumlah]).astype(np.float32)
    averaged = np.mean(stacked, axis=0).astype(np.uint8)
    psnr_value = PSNR(original_galaxy, averaged)
    results.append((jumlah, averaged, psnr_value))
    print(f'Average {jumlah:>3} citra -> PSNR = {psnr_value:.2f} dB')

fig, axes = plt.subplots(1, len(results), figsize=(3.2 * len(results), 3.5))
for ax, (jumlah, averaged, psnr_value) in zip(axes, results):
    ax.imshow(cv.cvtColor(averaged, cv.COLOR_BGR2RGB))
    ax.set_title(f'{jumlah} citra\nPSNR {psnr_value:.2f} dB')
    ax.axis('off')
plt.tight_layout()
plt.show()

**Tabel hasil (isi dengan angka PSNR dari output sel di atas):**

| No | Jumlah Citra di Average | Nilai PSNR (dB) |
|----|--------------------------|-------------------|
| 1  | 10                        |                   |
| 2  | 20                        |                   |
| 3  | 40                        |                   |
| 4  | 80                        |                   |
| 5  | 100                       |                   |

**Kesimpulan (isi berdasarkan angka PSNR di atas):** dari hasil percobaan, kenaikan PSNR paling besar terjadi di jumlah citra kecil (5 → 20), lalu mulai landai (diminishing return) di jumlah citra besar (40 → 100) — artinya menambah jumlah citra terus-menerus tidak selalu sepadan dengan biaya komputasinya. Sebutkan di sini pada jumlah berapa kenaikan PSNR pada punya Anda mulai tidak signifikan.

### 9. Image Masking

Citra asli: `images/couple.jpg` (sample sintetis pengganti `couple.tiff`). Mask: dua lingkaran putih. Dicoba beberapa operator logika: NOT, OR, AND, NAND, XOR.

In [ ]:
img = cv.imread('images/couple.jpg')
h, w = img.shape[:2]

mask = np.zeros((h, w), np.uint8)
cv.circle(mask, (w // 4, h // 2), min(h, w) // 5, 255, -1)
cv.circle(mask, (3 * w // 4, h // 2), min(h, w) // 5, 255, -1)
mask_3ch = cv.cvtColor(mask, cv.COLOR_GRAY2BGR)

operators = {
    'NOT (komplemen)': cv.bitwise_not(img),
    'OR (Atau)': cv.bitwise_or(img, mask_3ch),
    'AND (Dan)': cv.bitwise_and(img, mask_3ch),
    'NAND (Not And)': cv.bitwise_not(cv.bitwise_and(img, mask_3ch)),
    'XOR (Exclusive Or)': cv.bitwise_xor(img, mask_3ch),
}

fig, axes = plt.subplots(1, len(operators) + 1, figsize=(4 * (len(operators) + 1), 4))
axes[0].imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB)); axes[0].set_title('Input'); axes[0].axis('off')
for ax, (name, res) in zip(axes[1:], operators.items()):
    ax.imshow(cv.cvtColor(res, cv.COLOR_BGR2RGB))
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()

**Hasil analisa:**
- **AND**: hanya piksel di dalam kedua lingkaran (mask = 255) yang dipertahankan, sisanya jadi hitam (dikali 0) — dipakai untuk memotong/masking region of interest.
- **OR**: area mask (putih) memaksa hasil jadi putih penuh, area luar mask tetap citra asli — dipakai untuk menandai/menimpa region dengan warna solid.
- **NOT**: komplemen bit dari citra asli (warna "terbalik" secara biner, bukan sekedar invers 255-f seperti Tugas 1 karena ini operasi bitwise per-bit).
- **NAND**: kebalikan dari AND — area dalam lingkaran (hasil AND) dibalik, jadi area luar lingkaran yang justru tampil solid.
- **XOR**: karena citra asli bukan citra biner, XOR dengan mask 255 menghasilkan komplemen bit di dalam lingkaran dan citra asli tetap di luar lingkaran — polanya mirip AND/NOT tapi warnanya jadi "terbalik" hanya di dalam mask.

### 10. Foto Malam Hari

Pakai `images/night.jpg` (sample sintetis: citra digelapkan) sebagai pengganti foto malam hari asli. Metode yang dipilih: **Gamma Correction** dengan gamma < 1.

In [ ]:
night = cv.imread('images/night.jpg')
gamma = 0.4
normalized = night.astype(np.float32) / 255.0
brightened = np.clip(255 * (normalized ** gamma), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv.cvtColor(night, cv.COLOR_BGR2RGB)); axes[0].set_title('Foto malam (asli)')
axes[1].imshow(cv.cvtColor(brightened, cv.COLOR_BGR2RGB)); axes[1].set_title(f'Gamma Correction (γ = {gamma})')
plt.show()

**Alasan pemilihan metode:** Gamma Correction dipilih karena sifatnya non-linear — mengangkat detail di area gelap (shadow) jauh lebih kuat daripada area terang, sehingga highlight yang sudah cukup baik di foto malam hari tidak ikut over-expose seperti kalau pakai Linear Brightness (`+b` rata ke semua piksel).

**Risiko:** gamma kecil (<1) juga ikut menguatkan noise sensor yang biasanya dominan di area gelap pada foto malam hari, jadi hasil bisa terlihat grainy/berbintik kalau gamma-nya terlalu ekstrem.

### 11. Perbaikan Kualitas Citra `crayfish.jpg`

Pakai `images/crayfish.jpg` (sample sintetis: tekstur gelap+berisik dengan blob mirip crayfish) sebagai pengganti file asli di folder `Images`.

In [ ]:
crayfish = cv.imread('images/crayfish.jpg')

denoised = cv.fastNlMeansDenoisingColored(crayfish, None, h=8, hColor=8, templateWindowSize=7, searchWindowSize=21)
normalized = denoised.astype(np.float32) / 255.0
gamma = 0.7
enhanced = np.clip(255 * (normalized ** gamma), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(cv.cvtColor(crayfish, cv.COLOR_BGR2RGB)); axes[0].set_title('Before (asli)')
axes[1].imshow(cv.cvtColor(denoised, cv.COLOR_BGR2RGB)); axes[1].set_title('Denoising (Non-local Means)')
axes[2].imshow(cv.cvtColor(enhanced, cv.COLOR_BGR2RGB)); axes[2].set_title(f'Denoising + Gamma (γ = {gamma})')
plt.show()

- **Metode yang dipilih:** Denoising (Non-local Means, `cv.fastNlMeansDenoisingColored`) diikuti Gamma Correction.
- **Alasan pemilihan:** citra bawah air biasanya bernoise (partikel/sedimen) sekaligus gelap/low-contrast, jadi butuh dua tahap — denoise dulu supaya gamma correction tidak ikut menguatkan noise.
- **Parameter terbaik:** `h = hColor = 8`, `templateWindowSize = 7`, `searchWindowSize = 21` untuk denoising; `gamma = 0.7` untuk pencerahan. Nilai ini hasil coba-coba pada citra sample — sesuaikan lagi (`h` lebih besar = lebih halus tapi bisa hilang detail; `gamma` lebih kecil = lebih terang) begitu pakai citra `crayfish.jpg` asli dari modul.
- **Before-after:** ditampilkan pada 3 panel di atas (asli, setelah denoising, setelah denoising + gamma correction).